In [1]:
import os
import re
from pathlib import Path


# Import necessary libraries

from google.adk.models.lite_llm import LiteLlm # For OpenAI support


# Convenience libraries for working with Neo4j inside of Google ADK
from neo4j_for_adk import graphdb, tool_success, tool_error

from typing import Dict, Any
from dotenv import load_dotenv

import warnings
# Ignore all warnings
warnings.filterwarnings("ignore")

import logging
logging.basicConfig(level=logging.CRITICAL)

print("Libraries imported.")

Libraries imported.


In [2]:
load_dotenv()

MODEL_NAME = os.getenv("DEEPSEEK_MODEL")

llm = LiteLlm(MODEL_NAME)

print(llm.llm_client.completion(
    model=llm.model,
    messages=[{"role": "user", "content": "你准备好了吗？"}],
    tools=[],
))

print("\n Deepseek已经准备好了")

ModelResponse(id='a43c4b4b-0329-412e-9159-a1089e9c407b', created=1789536490, model='deepseek-flash', object='chat.completion', system_fingerprint='aeb56401ca74e127821c4f9126dcb669', choices=[Choices(finish_reason='stop', index=0, message=Message(content='准备好了！请告诉我你需要我做什么。', role='assistant', tool_calls=None, function_call=None, reasoning_content='我们需要回答用户中文：“你准备好了吗？” 需要作为助手回应。没有具体任务。应该简洁，可以问需要帮助什么。需要遵守。可以答准备好了，请告诉我需要做什么。也许要表达随时准备。最终用中文。', provider_specific_fields=None), provider_specific_fields={})], usage=Usage(completion_tokens=59, prompt_tokens=34, total_tokens=93, completion_tokens_details=CompletionTokensDetailsWrapper(accepted_prediction_tokens=None, audio_tokens=None, reasoning_tokens=50, rejected_prediction_tokens=None, text_tokens=None, image_tokens=None, video_tokens=None), prompt_tokens_details=PromptTokensDetailsWrapper(audio_tokens=None, cached_tokens=0, text_tokens=None, image_tokens=None, video_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=34))

 Deep

In [3]:
neo4j_is_ready = graphdb.send_query("RETURN 'Neo4j is Ready!' as message")

print(neo4j_is_ready)

{'status': 'error', 'error_message': 'Unable to retrieve routing information'}


In [4]:
from tools import load_product_nodes

# 加载 Product 类型的节点数据
load_product_nodes()


# 查询数据库中所有不是 __Entity__ 类型的节点标签
# 预期结果中应该能够找到 Product 标签
graphdb.send_query(
    """
    MATCH (n)
    WHERE NOT n:`__Entity__`
    RETURN DISTINCT labels(n) AS nonEntityLabels
    """
)

Libraries imported


{'status': 'error', 'error_message': 'Unable to retrieve routing information'}

In [5]:
# the approved construction plan should look something like this...
approved_construction_plan = {
    "Assembly": {
        "construction_type": "node",
        "source_file": "bom/assemblies.csv",
        "label": "Assembly",
        "unique_column_name": "assembly_id",
        "properties": ["assembly_name", "quantity", "product_id"]
    },
    "Part": {
        "construction_type": "node",
        "source_file": "bom/components.csv",
        "label": "Part",
        "unique_column_name": "part_id",
        "properties": ["part_name", "quantity", "assembly_id"]
    },
    "Product": {
        "construction_type": "node",
        "source_file": "bom/products.csv",
        "label": "Product",
        "unique_column_name": "product_id",
        "properties": ["product_name", "price", "description"]
    },
    "Supplier": {
        "construction_type": "node",
        "source_file": "bom/suppliers.csv",
        "label": "Supplier",
        "unique_column_name": "supplier_id",
        "properties": ["name", "specialty", "city", "country", "website", "contact_email"]
    },
    "Contains": {
        "construction_type": "relationship",
        "source_file": "bom/assemblies.csv",
        "relationship_type": "Contains",
        "from_node_label": "Product",
        "from_node_column": "product_id",
        "to_node_label": "Assembly",
        "to_node_column": "assembly_id",
        "properties": ["quantity"]
    },
    "Is_Part_Of": {
        "construction_type": "relationship",
        "source_file": "bom/components.csv",
        "relationship_type": "Is_Part_Of",
        "from_node_label": "Part",
        "from_node_column": "part_id",
        "to_node_label": "Assembly",
        "to_node_column": "assembly_id",
        "properties": ["quantity"]
    },
    "Supplied_By": {
        "construction_type": "relationship",
        "source_file": "bom/part_supplier_mapping.csv",
        "relationship_type": "Supplied_By",
        "from_node_label": "Part",
        "from_node_column": "part_id",
        "to_node_label": "Supplier",
        "to_node_column": "supplier_id",
        "properties": ["supplier_name", "lead_time_days", "unit_cost", "minimum_order_quantity", "preferred_supplier"]
    }
}



In [20]:
approved_files = [
    "product_reviews/gothenburg_table_reviews.md",
    "product_reviews/helsingborg_dresser_reviews.md",
    "product_reviews/jonkoping_coffee_table_reviews.md",
    "product_reviews/linkoping_bed_reviews.md",
    "product_reviews/malmo_desk_reviews.md",
    "product_reviews/norrkoping_nightstand_reviews.md",
    "product_reviews/orebro_lamp_reviews.md",
    "product_reviews/stockholm_chair_reviews.md",
    "product_reviews/uppsala_sofa_reviews.md",
    "product_reviews/vasteras_bookshelf_reviews.md"
]

In [7]:
# approved entities from the `ner_agent` of Lesson 7
approved_entities = ['Product', 'Issue', 'Feature', 'Location']

In [8]:
# approved fact types from the `relevant_fact_agent` of Lesson 7
approved_fact_types = {'has_issue': {'subject_label': 'Product', 'predicate_label': 'has_issue', 'object_label': 'Issue'}, 'includes_feature': {'subject_label': 'Product', 'predicate_label': 'includes_feature', 'object_label': 'Feature'}, 'used_in_location': {'subject_label': 'Product', 'predicate_label': 'used_in_location', 'object_label': 'Location'}}

In [9]:
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline


# 例如，创建一个知识图谱 Pipeline 需要提供下面这些参数
if False:
    example = SimpleKGPipeline(
        llm=None,  # 用于抽取实体和关系的 LLM
        driver=None,  # 用于将结果写入 Neo4j 图数据库的驱动
        embedder=None,  # 用于处理文本块的 Embedding 模型
        from_pdf=True,   # 这里近似设置为 True，因为将使用自定义加载器
        pdf_loader=None, # 用于加载 Markdown 的自定义加载器
        text_splitter=None, # 上面定义的文本分割器
        schema=None, # 刚刚定义的知识图谱 Schema
        prompt_template=None, # 用于对每个文本块执行实体抽取的 Prompt 模板
    )

In [10]:
from neo4j_graphrag.experimental.components.text_splitters.base import TextSplitter
from neo4j_graphrag.experimental.components.types import TextChunk, TextChunks


# 定义一个自定义文本切分器。
# 文本切分策略甚至可以由另一个 Agent 来决定。
class RegexTextSplitter(TextSplitter):
    """使用正则表达式匹配的分隔符来切分文本。"""

    def __init__(self, re: str):
        self.re = re

    async def run(self, text: str) -> TextChunks:
        """将一段文本切分成多个文本块。

        参数：
            text (str)：需要进行切分的文本。

        返回：
            TextChunks：由多个文本块组成的集合。
        """
        texts = re.split(self.re, text)
        i = 0
        chunks = [TextChunk(text=str(text), index=i) for (i, text) in enumerate(texts)]
        return TextChunks(chunks=chunks)

In [25]:
# 自定义文件数据加载器

from neo4j_graphrag.experimental.components.pdf_loader import DataLoader
from neo4j_graphrag.experimental.components.types import PdfDocument, DocumentInfo


class MarkdownDataLoader(DataLoader):

    def extract_title(self, markdown_text):
        # 定义一个用于匹配第一个一级标题（H1）的正则表达式
        pattern = r'^# (.+)$'

        # 在 Markdown 文本中搜索第一个匹配项
        match = re.search(pattern, markdown_text, re.MULTILINE)

        # 如果找到匹配项，则返回匹配到的标题；否则返回 "Untitled"
        return match.group(1) if match else "Untitled"

    async def run(self, filepath: Path, metadata = {}) -> PdfDocument:
        # 打开指定文件并读取全部内容
        with open(filepath, "r", encoding='utf-8') as f:
            markdown_text = f.read()

        # 从 Markdown 文本中提取文档标题
        doc_headline = self.extract_title(markdown_text)

        # 创建文档信息对象
        markdown_info = DocumentInfo(
            path=str(filepath),
            metadata={
                "title": doc_headline,
            }
        )

        # 将 Markdown 文本和文档信息包装成 PdfDocument 并返回
        return PdfDocument(
            text=markdown_text,
            document_info=markdown_info
        )

In [29]:
import os

from neo4j_graphrag.llm import OpenAILLM
from neo4j_graphrag.embeddings import OpenAIEmbeddings

# 使用 DeepSeek 作为 Neo4j GraphRAG 的 LLM
llm_for_neo4j = OpenAILLM(
    model_name="deepseek-v4-flash",
    model_params={
        "temperature": 0
    },
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

# 关键：告诉 GraphRAG 不要使用 Structured Output
llm_for_neo4j.supports_structured_output = False

# 使用阿里云百炼的 Embedding
embedder = OpenAIEmbeddings(
    model="qwen3.7-text-embedding",
    api_key=os.getenv("DASHSCOPE_API_KEY"),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# 使用原来的 Neo4j Driver
neo4j_driver = graphdb.get_driver()

In [13]:
### 8.3.5 实体 Schema

# 将前一个工作流中批准的实体类型用作实体提取的允许节点类型。这会限制大型语言模型（LLM）仅提取这些特定类型的实体。


# approved_entities 列表可以直接作为节点类型使用
schema_node_types = approved_entities

print("schema_node_types: ", schema_node_types)

# 通过提取谓词标签并将其转换为大写格式，将已批准的事实类型转换为关系类型，以满足模式的要求。

# approved_fact_types 字典的键可以直接作为关系类型，
# 并统一转换为大写格式
schema_relationship_types = [key.upper() for key in approved_fact_types.keys()]

print("schema_relationship_types: ", schema_relationship_types)

# 通过将事实类型转换为元组来创建关系模式，这些元组指定了特定节点类型之间允许的关系（主语-谓语-宾语模式）。

# 将 fact types 重新组织成元组列表
schema_patterns = [
    [fact['subject_label'], fact['predicate_label'].upper(), fact['object_label']]
    for fact in approved_fact_types.values()
]

print("schema_patterns:", schema_patterns)

# 构建完整的实体模式词典，该词典将指导大型语言模型（LLM）进行实体提取，将节点类型、关系类型和模式整合为单一配置。
# 完整的实体 Schema
entity_schema = {
    "node_types": schema_node_types,
    "relationship_types": schema_relationship_types,
    "patterns": schema_patterns,
    "additional_node_types": False,  # True 表示限制较弱，允许出现未知的节点类型
}

schema_node_types:  ['Product', 'Issue', 'Feature', 'Location']
schema_relationship_types:  ['HAS_ISSUE', 'INCLUDES_FEATURE', 'USED_IN_LOCATION']
schema_patterns: [['Product', 'HAS_ISSUE', 'Issue'], ['Product', 'INCLUDES_FEATURE', 'Feature'], ['Product', 'USED_IN_LOCATION', 'Location']]


该辅助函数从文件中提取前几行内容，为实体提取提供上下文。该上下文有助于大型语言模型在处理单个片段时更好地理解文档结构和内容。

In [22]:
def file_context(file_path: str, num_lines=5) -> str:
    """辅助函数：提取文件开头的几行内容

    参数：
        file_path (str)：文件路径
        num_lines (int，可选)：要提取的行数，默认值为 5。

    返回：
        str：文件开头的几行内容
    """

    # 打开文件并以只读方式读取
    with open(file_path, 'r', encoding='utf-8') as f:
        lines = []

        # 最多读取 num_lines 行
        for _ in range(num_lines):
            line = f.readline()

            # 如果已经到达文件末尾，则停止读取
            if not line:
                break

            # 将当前行添加到 lines 列表中
            lines.append(line)

    # 将所有读取到的行连接成一个字符串并返回
    return "\n".join(lines)

该函数用于生成用于实体和关系提取的上下文化提示模板。它将通用提取指令与文件特定的上下文相结合，以提高大型语言模型（LLM）对每个文本片段的实体识别准确率。

In [15]:
# 每个文本块的实体抽取 Prompt，并加入上下文
def contextualize_er_extraction_prompt(context: str) -> str:
    """创建一个带有文件前置内容上下文的 Prompt，用于实体和关系抽取。

    context 会被拼接到字符串中，
    这个字符串之后还会作为模板，
    用于填充 {schema} 和 {text} 等值。
    """

    general_instructions = """
    你是一种顶级的信息抽取算法，
    专门用于以结构化格式提取信息，以构建知识图谱。

    从下面的文本中提取实体（节点），并指定每个实体的类型。
    同时提取这些节点之间的关系。

    使用以下格式将结果返回为 JSON：
    {{"nodes": [ {{"id": "0", "label": "Person", "properties": {{"name": "John"}} }}],
    "relationships": [{{"type": "KNOWS", "start_node_id": "0", "end_node_id": "1", "properties": {{"since": "2024-08-01"}} }}] }}

    只能使用下面提供的节点类型和关系类型（如果有提供）：
    {schema}

    为每个节点分配一个唯一的 ID（字符串），
    并在定义关系时重复使用这个 ID。

    必须遵守关系的起点节点类型、终点节点类型以及关系方向。

    为确保生成有效的 JSON 对象，请遵守以下规则：
    - 除 JSON 之外，不要返回任何额外信息。
    - 不要在 JSON 外使用反引号，直接输出 JSON。
    - JSON 对象本身不能被包裹在列表中，它必须独立作为一个 JSON 对象。
    - 属性名称必须使用双引号括起来。
    """

    context_goes_here = f"""
    请考虑以下上下文，以帮助识别实体和关系：
    <context>
    {context}
    </context>"""

    input_goes_here = """
    输入文本：

    {text}
    """

    return general_instructions + "\n" + context_goes_here + "\n" + input_goes_here

### 8.4.1 创建 Neo4j KG Builder 管道

该函数通过提取文件上下文并生成基于上下文的提取提示，为特定文件创建一个定制化的KG构建器管道。它将所有先前定义的组件（加载器、分割器、模式、LLM）整合成一个完整的管道。

通过创建一个 KG 构建器管道并异步运行它，来处理每个已批准的 Markdown 文件。该过程会从文本片段中提取实体和关系，并将它们存储在 Neo4j 数据库中，作为主题图。

In [16]:
def make_kg_builder(file_path: str) -> SimpleKGPipeline:
    """为指定文件构建一个知识图谱构建器，
    用于为文本块切分和实体抽取提供上下文。"""

    # 读取文件开头的内容，作为上下文
    context = file_context(file_path)

    # 将文件上下文加入实体和关系抽取 Prompt
    contextualized_prompt = contextualize_er_extraction_prompt(context)

    # 返回配置完成的知识图谱构建 Pipeline
    return SimpleKGPipeline(
        llm=llm_for_neo4j, # 用于实体和关系抽取的 LLM
        driver=neo4j_driver,  # 用于将结果写入 Neo4j 图数据库的驱动
        embedder=embedder,  # 用于处理文本块的 Embedding 模型
        from_pdf=True,   # 这里近似设置为 True，因为将使用自定义加载器
        pdf_loader=MarkdownDataLoader(), # 用于加载 Markdown 的自定义加载器
        text_splitter=RegexTextSplitter("---"), # 前面定义的文本切分器
        schema=entity_schema, # 前面刚刚定义的知识图谱 Schema
        prompt_template=contextualized_prompt,
    )

In [30]:
from helper import get_neo4j_import_dir

# 获取 Neo4j 的 import 目录；如果没有获取到，则使用当前目录
neo4j_import_dir = get_neo4j_import_dir() or "."

# 遍历所有审核通过的文件
for file_name in approved_files:

    # 拼接 Neo4j import 目录和当前文件名，得到文件完整路径
    file_path = os.path.join(neo4j_import_dir, file_name)

    print(f"正在处理文件: {file_name}")

    # 根据当前文件创建知识图谱构建器
    kg_builder = make_kg_builder(file_path)

    # 异步运行知识图谱构建 Pipeline
    results = await kg_builder.run_async(file_path=str(file_path))

    # 输出当前文件的处理结果
    print("\t结果:", results.result)

print("所有文件处理完成。")

正在处理文件: product_reviews/gothenburg_table_reviews.md
	结果: {'resolver': {'number_of_nodes_to_resolve': 59, 'number_of_created_nodes': 50}}
正在处理文件: product_reviews/helsingborg_dresser_reviews.md
	结果: {'resolver': {'number_of_nodes_to_resolve': 126, 'number_of_created_nodes': 112}}
正在处理文件: product_reviews/jonkoping_coffee_table_reviews.md
	结果: {'resolver': {'number_of_nodes_to_resolve': 170, 'number_of_created_nodes': 155}}
正在处理文件: product_reviews/linkoping_bed_reviews.md
	结果: {'resolver': {'number_of_nodes_to_resolve': 212, 'number_of_created_nodes': 198}}
正在处理文件: product_reviews/malmo_desk_reviews.md
	结果: {'resolver': {'number_of_nodes_to_resolve': 252, 'number_of_created_nodes': 235}}
正在处理文件: product_reviews/norrkoping_nightstand_reviews.md
	结果: {'resolver': {'number_of_nodes_to_resolve': 294, 'number_of_created_nodes': 273}}
正在处理文件: product_reviews/orebro_lamp_reviews.md
	结果: {'resolver': {'number_of_nodes_to_resolve': 336, 'number_of_created_nodes': 319}}
正在处理文件: product_reviews/stock

In [27]:
import inspect
from neo4j_graphrag.experimental.pipeline.kg_builder import SimpleKGPipeline

print(inspect.signature(SimpleKGPipeline))
print(inspect.signature(llm_for_neo4j.invoke))
print(inspect.signature(llm_for_neo4j.ainvoke))

(llm: 'LLMInterface', driver: 'neo4j.Driver', embedder: 'Embedder', entities: 'Optional[Sequence[EntityInputType]]' = None, relations: 'Optional[Sequence[RelationInputType]]' = None, potential_schema: 'Optional[List[tuple[str, str, str]]]' = None, schema: "Optional[Union[GraphSchema, dict[str, list[Any]], Literal['FREE', 'EXTRACTED']],]" = None, from_file: 'bool' = True, from_pdf: 'Optional[bool]' = None, text_splitter: 'Optional[TextSplitter]' = None, file_loader: 'Optional[DataLoader]' = None, pdf_loader: 'Optional[DataLoader]' = None, kg_writer: 'Optional[KGWriter]' = None, on_error: 'str' = 'IGNORE', prompt_template: 'Union[ERExtractionTemplate, str]' = <neo4j_graphrag.generation.prompts.ERExtractionTemplate object at 0x000001ADEAEC8CD0>, perform_entity_resolution: 'bool' = True, lexical_graph_config: 'Optional[LexicalGraphConfig]' = None, neo4j_database: 'Optional[str]' = None)
(input: 'Union[str, List[LLMMessage]]', message_history: 'Optional[Union[List[LLMMessage], MessageHistor

## 实体解析的工具定义


将主题图中的实体与领域图中的实体建立关联。

对于主题图中的每种实体类型，你需要制定一种策略，将其
与领域图中的相应节点建立关联。

例如，应预期主题图中存在具有产品名称的“产品”实体，
且这些实体应与领域图中的产品建立关联。

为此，您需要：
1. 查找主题图中的唯一实体标签
2. 查找领域图中的唯一节点标签
3. 尝试关联属性键
4. 通过分析属性值的相似性来执行实体解析

In [31]:
# 第一：先查看实体标签
# 查询 Neo4j 数据库，找出所有带有 `__Entity__` 标签的节点（由知识图谱构建器创建的实体），并返回它们的唯一标签组合。
results = graphdb.send_query("""MATCH (n)
    WHERE n:`__Entity__`
    RETURN DISTINCT labels(n) AS entity_labels
    """)

results['query_result']


# 将这些标签列表展开成单独的标签
# 使用 UNWIND 将标签数组展平为单独的标签字符串，该操作会将标签数组转换为每个标签对应的一行。
results = graphdb.send_query("""MATCH (n)
    WHERE n:`__Entity__`
    WITH DISTINCT labels(n) AS entity_labels
    UNWIND entity_labels AS entity_label
    RETURN DISTINCT entity_label
    """)

results['query_result']


# 过滤掉以 "__" 开头的标签
# 过滤掉以双下划线（“__”）开头的内部 Neo4j 标签，以便仅关注从文本中提取的有意义的实体类型标签。
results = graphdb.send_query("""MATCH (n)
    WHERE n:`__Entity__`
    WITH DISTINCT labels(n) AS entity_labels
    UNWIND entity_labels AS entity_label
    WITH entity_label
    WHERE NOT entity_label STARTS WITH "__"
    RETURN entity_label
    """)

results['query_result']


# 将查询封装成一个可以调用的函数
# 将前面的查询步骤整合成一个可重用的函数，该函数返回主题图中所有唯一的实体标签，并排除 Neo4j 内部系统标签。
def find_unique_entity_labels():
    result = graphdb.send_query("""MATCH (n)
        WHERE n:`__Entity__`
        WITH DISTINCT labels(n) AS entity_labels
        UNWIND entity_labels AS entity_label
        WITH entity_label
        WHERE NOT entity_label STARTS WITH "__"
        RETURN collect(entity_label) as unique_entity_labels
        """)
    if result['status'] == 'error':
        raise Exception(result['message'])
    return result['query_result'][0]['unique_entity_labels']


# 测试这个函数
unique_entity_labels = find_unique_entity_labels()

print("Unique entity labels: ", unique_entity_labels)

Unique entity labels:  ['Product', 'Feature', 'Location', 'Issue']


In [32]:
def find_unique_domain_keys(domainLabel: str):
    """查找指定领域节点标签所包含的所有唯一属性键。"""

    result = graphdb.send_query(
        """MATCH (n:$($domainLabel))
        WHERE NOT n:`__Entity__` // 排除由知识图谱构建器创建的实体节点，这些节点应该是领域节点
        WITH DISTINCT keys(n) as domainKeys
        UNWIND domainKeys as domainKey
        RETURN collect(distinct(domainKey)) as unique_domain_keys
        """,
        {
            "domainLabel": domainLabel
        }
    )

    # 如果查询出错，则抛出异常
    if result['status'] == 'error':
        raise Exception(result['message'])

    return result['query_result'][0]['unique_domain_keys']


find_unique_domain_keys("Product")

[]

In [ ]:
def find_unique_entity_keys(entityLabel: str):
    """查找指定实体标签节点所包含的所有唯一属性键。"""

    result = graphdb.send_query("""MATCH (n:$($entityLabel))
    WHERE n:`__Entity__`
    WITH DISTINCT keys(n) as entityKeys
    UNWIND entityKeys as entityKey
    RETURN collect(distinct(entityKey)) as unique_entity_keys
    """, {
        "entityLabel": entityLabel
    })

    # 如果查询出错，则抛出异常
    if result['status'] == 'error':
        raise Exception(result['message'])

    # 返回唯一的实体属性键列表
    return result['query_result'][0]['unique_entity_keys']


# 测试函数：
# 获取标签为 Product 的实体节点所包含的唯一属性键
find_unique_entity_keys("Product")

In [34]:
def normalize_key(label: str, key: str) -> str:
    """针对指定标签，将属性键标准化。

    属性键标准化规则：
    - 将属性键转换为小写
    - 删除属性键开头和结尾的空白字符
    - 删除属性键中的标签前缀
    - 将属性键内部的空格替换为 "_"

    例如：
        - "Product_name" -> "name"
        - "product name" -> "name"
        - "Price" -> "price"

    参数：
        label (str)：要进行属性键标准化的节点标签
        key (str)：需要进行标准化的属性键

    返回：
        str：标准化后的属性键
    """

    # 将属性键转换为小写
    lowercase_key = key.lower()

    # 删除属性键开头的 label 前缀，以及前缀后可能存在的 "_" 或空格
    unprefixed_key = re.sub(
        f"^{label.lower()}[_ ]*",
        "",
        lowercase_key
    )

    # 将属性键内部的空格替换为 "_"
    normalized_key = re.sub(" ", "_", unprefixed_key)

    # 返回标准化后的属性键
    return normalized_key


print(normalize_key("Product", "Product_name"))
print(normalize_key("Product", "Product Name"))
print(normalize_key("Product", "product name"))
print(normalize_key("Product", "Price"))

name
name
name
price


In [ ]:
# 使用 rapidfuzz 库进行模糊文本相似度评分
from rapidfuzz import fuzz


# 对于指定的 label，获取相互关联的实体属性键和领域属性键
def correlate_entity_and_domain_keys(
    label: str,
    entity_keys: list[str],
    domain_keys: list[str],
    similarity: float = 0.9
) -> list[tuple[str, str, float]]:

    correlated_keys = []

    for entity_key in entity_keys:
        for domain_key in domain_keys:

            # 目前只根据归一化后的键进行匹配；这里也可以进一步使用模糊匹配
            normalized_entity_key = normalize_key(label, entity_key)
            normalized_domain_key = normalize_key(label, domain_key)

            # rapidfuzz 的相似度范围是 0.0 -> 100.0，
            # 因此除以 100 转换为 0.0 -> 1.0
            fuzzy_similarity = (
                fuzz.ratio(
                    normalized_entity_key,
                    normalized_domain_key
                ) / 100
            )

            if (fuzzy_similarity > similarity):
                correlated_keys.append(
                    (entity_key, domain_key, fuzzy_similarity)
                )

    # 按相似度从高到低排序
    correlated_keys.sort(
        key=lambda x: x[2],
        reverse=True
    )

    return correlated_keys


label = "Product"

entity_keys = find_unique_entity_keys(label)

domain_keys = find_unique_domain_keys(label)


# 尝试使用一个相对较低的相似度阈值
correlated_keys = correlate_entity_and_domain_keys(
    label,
    entity_keys,
    domain_keys,
    similarity=0.5
)


print(
    f"{label} correlated keys "
    "(entity key, domain key, similarity score)..."
)


# 查看匹配到的属性键
correlated_keys